In [1]:
import pandas as pd
import numpy as np

from data_preprocessingagent import DataPreprocessingAgent

In [2]:
pre_agent=DataPreprocessingAgent()
train,test,rul=pre_agent.preprocess(
    "train_FD003.txt",
    "test_FD003.txt",
    "RUL_FD003.txt"
)
print(train.shape)
train.head()

Shape: (24720, 26)
Missing: 0
Duplicates: 0
(24720, 21)


,engine_id,cycle,setting1,setting2,sensor2,sensor3,sensor4,sensor6,sensor7,sensor8,...,sensor10,sensor11,sensor12,sensor13,sensor14,sensor15,sensor17,sensor20,sensor21,RUL
0,1,1,0.470930,0.769231,0.355972,0.370523,0.308580,1.0,0.208812,0.623529,...,0.333333,0.348571,0.231279,0.642857,0.239116,0.647755,0.272727,0.559524,0.446331,125
1,1,2,0.546512,0.230769,0.388759,0.399100,0.309360,1.0,0.236590,0.647059,...,0.333333,0.308571,0.236882,0.654762,0.278567,0.685659,0.363636,0.488095,0.534836,125
2,1,3,0.418605,0.307692,0.313817,0.353298,0.445398,1.0,0.230843,0.664706,...,0.333333,0.302857,0.217015,0.636905,0.264526,0.564462,0.272727,0.404762,0.458577,125
3,1,4,0.383721,0.538462,0.487119,0.417107,0.237285,1.0,0.268199,0.647059,...,0.333333,0.314286,0.240448,0.684524,0.245612,0.558909,0.363636,0.470238,0.391966,125
4,1,5,0.593023,0.461538,0.196721,0.476218,0.321217,1.0,0.245690,0.670588,...,0.333333,0.262857,0.245033,0.654762,0.252109,0.556736,0.363636,0.577381,0.543371,125


In [3]:
print(train.columns.tolist())

['engine_id', 'cycle', 'setting1', 'setting2', 'sensor2', 'sensor3', 'sensor4', 'sensor6', 'sensor7', 'sensor8', 'sensor9', 'sensor10', 'sensor11', 'sensor12', 'sensor13', 'sensor14', 'sensor15', 'sensor17', 'sensor20', 'sensor21', 'RUL']


In [4]:
sensor_cols=[
    col for col in train.columns
    if col.startswith("sensor")
]
print(sensor_cols)

['sensor2', 'sensor3', 'sensor4', 'sensor6', 'sensor7', 'sensor8', 'sensor9', 'sensor10', 'sensor11', 'sensor12', 'sensor13', 'sensor14', 'sensor15', 'sensor17', 'sensor20', 'sensor21']


### Compute rolling averages

In [5]:
window_size=10
for sensor in sensor_cols:
    train[f"{sensor}_trend"]=(
        train.groupby("engine_id")[sensor].transform(
            lambda x:
            x.rolling(window=window_size,min_periods=1).mean()
        )
    )

### Degradation score

In [11]:
#### Healthy baseline = first cycle of each engine.

In [12]:
baseline=(train.groupby("engine_id").first()[sensor_cols])

In [13]:
degradation_scores=[]
for idx,row in train.iterrows():
    engine=row["engine_id"]
    baseline_values=baseline.loc[engine]
    current_values=row[sensor_cols]
    degradation=np.mean(np.abs(current_values.values-baseline_values.values))
    degradation_scores.append(degradation)
train["Degradation_Score"]=degradation_scores

### Normalize the degradation score

In [14]:
max_deg=train["Degradation_Score"].max()
train["Degradation_Health"]=(100-(train["Degradation_Score"]/max_deg)*100)
train["Degradation_Health"]=(train["Degradation_Health"].clip(lower=0))

### Anomaly Detection

In [16]:
anomaly_count=[]
for sensor in sensor_cols:
    zscore=(train[sensor]-train[sensor].mean())/train[sensor].std()
    train[f"{sensor}_anomaly"]=(np.abs(zscore)>3).astype(int)

In [17]:
#count the anomalies
anomaly_cols=[
    c for c in train.columns
    if "_anomaly" in c
]
train["Anomaly_Count"]=(train[anomaly_cols].sum(axis=1))

### RUL Health score

In [19]:
train["RUL_Health"]=(train["RUL"]/125)*100
train["RUL_Health"]=(train["RUL_Health"].clip(0,100))

### Final Health Score

Combine degradation + anomaly + RUL.

Weights:

40% Degradation
40% RUL
20% Anomaly

In [20]:
train["Anomaly_Health"]=(100-(train["Anomaly_Count"]/train["Anomaly_Count"].max())*100)
train["Anomaly_Health"]=(train["Anomaly_Health"].fillna(100))

In [21]:
train["Health_Score"]=(
    0.4*train["Degradation_Health"]+
    0.4*train["RUL_Health"]+
    0.2 * train["Anomaly_Health"])

### Health Status

In [22]:
def get_status(score):
    if score>=70:
        return "Healthy"
    elif score>=40:
        return "Degrading"
    else:
        return "Critical"

In [23]:
train["Health_Status"]=(train["Health_Score"].apply(get_status))

### Alert Generation

In [24]:
def generate_alert(row):
    if row["Health_Status"]=="Critical":
        return("Immediate inspection required")
    elif row["Health_Status"]=="Degrading":
        return("Monitor engine closely")
    else:
        return("Normal operation")

In [25]:
train["Alert"]=(train.apply(generate_alert,axis=1))

### View Results

In [26]:
train[
    [
        "engine_id",
        "cycle",
        "RUL",
        "Health_Score",
        "Health_Status",
        "Alert"
    ]
].head(20)

,engine_id,cycle,RUL,Health_Score,Health_Status,Alert
0,1,1,125,100.000000,Healthy,Normal operation
1,1,2,125,97.655816,Healthy,Normal operation
2,1,3,125,97.013170,Healthy,Normal operation
3,1,4,125,96.452752,Healthy,Normal operation
4,1,5,125,96.035272,Healthy,Normal operation
5,1,6,125,96.508289,Healthy,Normal operation
6,1,7,125,96.836692,Healthy,Normal operation
7,1,8,125,97.507475,Healthy,Normal operation
8,1,9,125,96.096874,Healthy,Normal operation
9,1,10,125,96.960142,Healthy,Normal operation


### Save Output

In [27]:
train.to_csv("health_monitoring_output.csv",index=False)
print("Health Monitoring Output Saved")

Health Monitoring Output Saved
